In [2]:
import urllib.request
import zipfile
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

def download_dataset():
    """Downloads and extracts the UCI SMS Spam Collection dataset."""
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
    zip_path = "smsspamcollection.zip"
    data_path = "SMSSpamCollection"

    # Download and extract if it doesn't already exist
    if not os.path.exists(data_path):
        print("Downloading SMS Spam Collection dataset...")
        urllib.request.urlretrieve(url, zip_path)
        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(".")
        os.remove(zip_path) # Clean up the zip file

    print("Dataset ready.")
    return data_path

def load_and_preprocess_data(file_path):
    """Loads the dataset into a pandas DataFrame."""
    # The SMS Spam Collection is a tab-separated file with no header
    df = pd.read_csv(file_path, sep='\t', header=None, names=['label', 'text'])
    return df

def train_and_compare_models(df):
    """Trains both Naive Bayes and Logistic Regression to compare them."""
    X = df['text']
    y = df['label']

    # Split the dataset 75% for training and 25% for testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # 1. Naive Bayes Pipeline (Great baseline for text)
    nb_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('classifier', MultinomialNB())
    ])

    # 2. Logistic Regression Pipeline (Often performs better with robust feature representation)
    lr_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english')),
        ('classifier', LogisticRegression(max_iter=1000))
    ])

    print("\n--- Training Naive Bayes ---")
    nb_pipeline.fit(X_train, y_train)
    nb_predictions = nb_pipeline.predict(X_test)
    print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_predictions):.4f}")

    print("\n--- Training Logistic Regression ---")
    lr_pipeline.fit(X_train, y_train)
    lr_predictions = lr_pipeline.predict(X_test)
    print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_predictions):.4f}")

    print("\nLogistic Regression Classification Report:")
    print(classification_report(y_test, lr_predictions))

    return lr_pipeline

if __name__ == "__main__":
    # 1. Download data
    data_path = download_dataset()

    # 2. Load data
    df = load_and_preprocess_data(data_path)

    print(f"\nLoaded dataset with {len(df)} records.")
    print("Label distribution:")
    print(df['label'].value_counts())

    # 3. Train models and get the Logistic Regression model
    best_model = train_and_compare_models(df)

    # 4. Interactive testing loop
    print("\n--- Try it out! ---")
    print("Type your message below to classify it. Type 'quit' or 'exit' to stop.")

    while True:
        try:
             email = input("\nEnter message: ")
             if email.strip().lower() in ['quit', 'exit']:
                 break
             if not email.strip():
                 continue

             prediction = best_model.predict([email])[0]
             print(f"Prediction: {prediction.upper()}")
        except KeyboardInterrupt:
             print("\nExiting...")
             break
        except EOFError:
             break


Extracting dataset...
Dataset ready.

Loaded dataset with 5572 records.
Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

--- Training Naive Bayes ---
Naive Bayes Accuracy: 0.9742

--- Training Logistic Regression ---
Logistic Regression Accuracy: 0.9677

Logistic Regression Classification Report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98      1207
        spam       1.00      0.76      0.86       186

    accuracy                           0.97      1393
   macro avg       0.98      0.88      0.92      1393
weighted avg       0.97      0.97      0.97      1393


--- Try it out! ---
Type your message below to classify it. Type 'quit' or 'exit' to stop.

Enter message: IMPORTANT: Your Microsoft account password will expire today. Click to reset."
Prediction: HAM

Enter message: Enlarge your business potential with our new marketing strategies!"
Prediction: HAM

Enter message: Your package could not be 